### 이슈 분석

In [15]:
# 라이브러리 불러오기
import json
import time
from datetime import datetime
from collections import Counter
from openai import OpenAI
import pandas as pd

# 모델 설정 (나중에 Claude로 바꿀 때 여기만 변경)
MODEL = "gemma4:e4b"

# Ollama 클라이언트
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

# 연결 확인
try:
    models = ollama.models.list()
    print("Ollama 연결 성공")
    print("사용 가능한 모델:")
    for m in models.data:
        print(f"  - {m.id}")
except Exception as e:
    print(f"Ollama 연결 실패: {e}")
    print("터미널에서 'ollama serve' 실행했는지 확인해줘요!")

Ollama 연결 성공
사용 가능한 모델:
  - gemma4:e4b
  - mistral:7b
  - gemma3:4b


In [12]:
# 데이터 로드
with open("../data/raw_articles.json", "r", encoding="utf-8") as f:
    all_articles = json.load(f)

# flatten
articles = []
for category, items in all_articles.items():
    for item in items:
        item['category'] = category
        articles.append(item)

print(f"총 {len(articles)}개의 기사 로드됨")

총 1295개의 기사 로드됨


In [ ]:
# 중복 제거
def deduplicate(articles: list) -> list:
    seen = set()
    unique = []
    for article in articles:
        h = article.get('hash', '')
        if h not in seen:
            seen.add(h)
            unique.append(article)
    print(f'중복제거: {len(articles)}개 -> {len(unique)}개')
    return unique

articles = deduplicate(articles)

중복제거: 1295개 -> 1267개


In [18]:
# 필터링

def filter_article(article: dict) -> bool:
    '''관련 없는 기사 빠르게 제거'''
    try:
        response = ollama.chat.completions.create(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": f"""다음 기사가 경제/정치/부동산/테크/금융 중 하나라도 관련있으면 YES, 완전히 관련없으면 NO만 답해. 다른 말 하지마.
제목: {article['title']}
답:"""
            }],
            max_tokens=5,
            temperature=0.1
        )
        answer = response.choices[0].message.content.strip().upper()
        return "YES" in answer
    except Exception as e:
        return True  # 에러 시 포함

# 테스트 먼저
test = filter_article(articles[0])
print(f"테스트: {articles[0]['title'][:40]} → {'포함' if test else '제외'}")

테스트: "역대급 더위 온다"…삼성전자, 에어컨 생산라인 풀가동 → 포함


In [17]:
# 전체 필터링 실행
print(f'필터링 시작 ({len(articles)}개)')
start = time.time()

filtered = []
for i, article in enumerate(articles):
    result = filter_article(article)
    if result:
        filtered.append(article)
    if i % 100 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/{len(articles)}개 처리 ({elapsed:.0f}초 경과)")

print(f"필터링 완료: {len(articles)}개 → {len(filtered)}개")
print(f"소요 시간: {time.time()-start:.0f}초")

필터링 시작 (1295개)
  1/1295개 처리 (1초 경과)
  101/1295개 처리 (52초 경과)
  201/1295개 처리 (103초 경과)
  301/1295개 처리 (152초 경과)
  401/1295개 처리 (201초 경과)
  501/1295개 처리 (253초 경과)
  601/1295개 처리 (297초 경과)
  701/1295개 처리 (342초 경과)
  801/1295개 처리 (390초 경과)
  901/1295개 처리 (439초 경과)
  1001/1295개 처리 (489초 경과)
  1101/1295개 처리 (535초 경과)
  1201/1295개 처리 (579초 경과)
필터링 완료: 1295개 → 380개
소요 시간: 622초


In [19]:
# 스코어링
def score_article(article: dict) -> dict:
    '''중요도 스코어링 + 한줄 요약'''
    try:
        response = ollama.chat.completions.create(
            model=MODEL,
            messages=[{
                'role': 'user',
                'content': f'''다음 뉴스를 분석해줘. 반드시 JSON만 답해. 다른 텍스트 없이.

제목: {article['title']}
요약: {article.get('summary', '')[:200]}

{{'importance': 1~10, 'one_line': '한줄요약', 'is_urgent': true또는false, 'keywords': ['키워드1', '키워드2']}}'''
            }],
            max_tokens=200,
            temperature=0.1
        )
        text = response.choices[0].message.content.strip()
        start_idx = text.find('{')
        end_idx = text.rfind('}') + 1
        if start_idx != -1:
            return json.loads(text[start_idx:end_idx])
    except:
        pass
    return {'importance': 3, 'one_line': article['title'], 'is_urgent': False, 'keywords': []}

# 테스트
test = score_article(filtered[0])
print(json.dumps(test, ensure_ascii=False, indent=2))

{
  "importance": 3,
  "one_line": "중동發 에너지 위기에도…\"재생에너지로의 즉각 전환 어려워\"",
  "is_urgent": false,
  "keywords": []
}


In [21]:
# 전체 스코어링 + 상위 선별
print(f'스코어링 시작 ({len(filtered)}건)')
start = time.time()

for i, article in enumerate(filtered):
    score = score_article(article)
    article['score'] = score
    if (i + 1) % 50 == 0:
        print(f'{i+1}/{len(filtered)}완료')
        
print(f'스코어링 완료 ({time.time() - start:.1f}초)')

# 중요도 분포
dist = Counter(a.get('importance', 0) for a in filtered)
print('중요도 분포:')
for score in sorted(dist.keys(), reverse=True):
    flag = "🔴" if score >= 8 else "🟡" if score >= 6 else "⚪"
    print(f"  {flag} {score}점: {dist[score]}개")

# 카테고리별 상위 8개 선별
top_articles = []
for category in ['politics_economy', 'global_macro', 'tech_industry', 'real_estate']:
    cat = sorted(
        [a for a in filtered if a['category'] == category],
        key=lambda x: x.get('importance', 0),
        reverse=True
    )[:8]
    top_articles.extend(cat)

# 긴급 이슈 추가
urgent = [a for a in filtered if a.get('is_urgent') and a not in top_articles]
top_articles.extend(urgent)

print(f'최종 선별: {len(top_articles)}개 (긴급 {len(urgent)}개 포함)')

스코어링 시작 (380건)
50/380완료
100/380완료
150/380완료
200/380완료
250/380완료
300/380완료
350/380완료
스코어링 완료 (2900.1초)
중요도 분포:
  ⚪ 0점: 380개
최종 선별: 32개 (긴급 0개 포함)


In [ ]:
# 심층분석
def analyze_article(article: dict) -> dict:
    try:
        response = ollama.chat.completions.create(
            model=MODEL,
            messages=[{
                'role': 'user',
                'content': f'''Analyze this Korean news article. Respond in JSON only. No other text. No markdown.

Title: {article['title']}
Summary: {article.get('summary', '')[:300]}
Category: {article['category']}

Respond exactly in this format:
{{"issue": "one line in Korean", "cause": "cause in Korean", "economic_impact": "3 sentences in Korean", "affected_sectors": ["sector1", "sector2"], "sentiment": "positive", "prediction": "prediction in Korean", "importance": 5}}'''
            }],
            max_tokens=2048,
            temperature=0.1
        )
        text = response.choices[0].message.content.strip()
        text = text.replace('```json', '').replace('```', '').strip()
        print('raw:', text[:100])
        start_idx = text.find('{')
        end_idx = text.rfind('}') + 1
        if start_idx != -1:
            result = json.loads(text[start_idx:end_idx])
            result['title'] = article['title']
            result['link'] = article.get('link', '')
            result['category'] = article['category']
            return result
    except Exception as e:
        print(f'  실패: {e}')
    return None

test = analyze_article(top_articles[0])
print(json.dumps(test, ensure_ascii=False, indent=2))

raw: {"issue": "중동발 에너지 위기에도 재생에너지로의 즉각적 전환에 어려움이 존재함", "cause": "기존 에너지 시스템의 구조적 한계와 대규모 인프라 구축 및 안정화의 어
{
  "issue": "중동발 에너지 위기에도 재생에너지로의 즉각적 전환에 어려움이 존재함",
  "cause": "기존 에너지 시스템의 구조적 한계와 대규모 인프라 구축 및 안정화의 어려움",
  "economic_impact": "에너지 가격의 급등락은 산업 전반의 생산 비용을 증가시키고 경제 불확실성을 높이고 있다. 단기적으로는 화석 연료 의존도가 높아져 물가 상승 압력을 가중시킬 수 있다. 따라서 에너지 안보 확보와 경제 안정화를 위한 다각적인 정책 마련이 시급하다.",
  "affected_sectors": [
    "에너지",
    "제조업",
    "운송업"
  ],
  "sentiment": "negative",
  "prediction": "단기적으로는 화석 연료와 재생에너지를 혼합하는 '믹스 에너지' 전략이 불가피하며, 정부의 적극적인 정책 개입이 요구된다.",
  "importance": 5,
  "title": "중동發 에너지 위기에도…\"재생에너지로의 즉각 전환 어려워\"",
  "link": "https://www.hankyung.com/article/2026041248427",
  "category": "politics_economy"
}


In [39]:
# 전체 심층 분석
print(f'심층 분석 시작 ({len(top_articles)}개)...')
start = time.time()

analyzed = []
for i, article in enumerate(top_articles):
    print(f'  {i+1}/{len(top_articles)} - {article["title"][:40]}')
    result = analyze_article(article)
    if result:
        analyzed.append(result)
    time.sleep(0.1)

print(f'\n분석 완료: {len(analyzed)}개 ({time.time()-start:.0f}초)')

urgent_results = [a for a in analyzed if a.get('importance', 0) >= 8]
print(f'긴급 이슈: {len(urgent_results)}개')
for a in urgent_results:
    print(f'  - {a["title"][:50]}')

심층 분석 시작 (32개)...
  1/32 - 중동發 에너지 위기에도…"재생에너지로의 즉각 전환 어려워"
raw: {"issue": "중동발 에너지 위기 속 재생에너지 전환의 어려움", "cause": "중동 지역의 지정학적 불안정 및 에너지 공급망 충격", "economic_impact": 
  2/32 - 금융위, 우수 직원에 최대 1000만원 상금…'명예의 전당'도 만든다 [
raw: {"issue": "금융위가 우수 직원 포상 및 명예의 전당 제도를 도입하여 조직 문화 개선을 추진한다.", "cause": "금융기관의 조직 문화 활성화 및 직원들의 사기 진작을
  3/32 - 프랜차이즈 가맹본부 13.2% 증가…주춤했던 가맹산업 다시 활기
raw: {"issue": "프랜차이즈 가맹본부 증가세로 가맹산업 활력 회복", "cause": "소비 심리 회복과 외식 시장의 지속적인 성장세", "economic_impact": "가맹
  4/32 - 국세청장 "법인 보유 고가주택 2360개 점검…사주일가 살면 탈세"
raw: {"issue": "법인이 보유한 고가 주택을 이용한 세금 탈세 및 자산 오용 문제", "cause": "법인 자산을 사적 용도로 사용하며 세금 회피를 시도하는 불투명한 자산 운용
  5/32 - 삼성전자, 中사업 구조조정 시작하나…"반도체 분야 집중" 전망
raw: {"issue": "삼성전자의 중국 사업 구조조정 및 반도체 분야 집중 전략", "cause": "미중 기술 패권 경쟁 심화와 지정학적 리스크 증가에 따른 사업 효율화 필요성", 
  6/32 - 석유 등 공급망 안정에 '1.1조' 투입…"민생물가 안정 기대"
raw: {"issue": "석유 등 핵심 공급망 안정화에 1.1조 원 투입", "cause": "글로벌 공급망 불안정과 에너지 가격 급등으로 인한 물가 및 경제 불안정 심화", "econ
  7/32 - 27일부터 '최대 60만원' 고유가 지원금…신청 방법은?
raw: {"issue": "고유가에 대응하기 위한 정부의

In [40]:
with open("../data/analyzed_articles.json", "w", encoding="utf-8") as f:
    json.dump(analyzed, f, ensure_ascii=False, indent=2)

print("저장 완료: data/analyzed_articles.json")

# 미리보기
df = pd.DataFrame(analyzed)[["title", "issue", "sentiment", "importance"]]
print(df.head(10).to_string())

저장 완료: data/analyzed_articles.json
                                                                                                               title                                            issue sentiment  importance
0                                                                                   중동發 에너지 위기에도…"재생에너지로의 즉각 전환 어려워"                       중동발 에너지 위기 속 재생에너지 전환의 어려움  negative           5
1                                                                 금융위, 우수 직원에 최대 1000만원 상금…'명예의 전당'도 만든다 [금융당국 백브리핑]  금융위가 우수 직원 포상 및 명예의 전당 제도를 도입하여 조직 문화 개선을 추진한다.  positive           5
2                                                                                프랜차이즈 가맹본부 13.2% 증가…주춤했던 가맹산업 다시 활기                       프랜차이즈 가맹본부 증가세로 가맹산업 활력 회복  positive           5
3                                                                              국세청장 "법인 보유 고가주택 2360개 점검…사주일가 살면 탈세"              법인이 보유한 고가 주택을 이용한 세금 탈세 및 자산 오용 문제  negative           5
4                        